In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/GeoTwin_Lusaka")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

In [ ]:
# Install dependencies
!pip -q install rasterio geopandas shapely pandas numpy matplotlib sqlalchemy geoalchemy2 psycopg2-binary plotly kaleido

print("GeoTwin Lusaka libraries installed successfully.")

In [ ]:
#Download the dataset
import requests
from pathlib import Path

DATA_URL = "https://opendata.aiforgood.ai/building-density/locations/lusaka_district/2025q2_cog.tif"
RASTER_PATH = PROJECT_DIR / "Lusaka_Building_Density_Height.tif"

response = requests.get(DATA_URL, stream=True, timeout=120)
response.raise_for_status()

with open(RASTER_PATH, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Download completed.")
print("Saved to:", RASTER_PATH)
print("File size:", round(RASTER_PATH.stat().st_size / (1024 * 1024), 2), "MB")

In [ ]:
# Inspect the raster
import rasterio

with rasterio.open(RASTER_PATH) as src:
    print("CRS:", src.crs)
    print("Dimensions:", src.width, "×", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Number of bands:", src.count)
    print("Data type:", src.dtypes)
    print("NoData value:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Band units:", src.units)
    print("Metadata:", src.tags())

    for band in range(1, src.count + 1):
        values = src.read(band, masked=True)
        print(f"\nBand {band}")
        print("Valid pixels:", values.count())
        print("Minimum:", values.min())
        print("Maximum:", values.max())

In [ ]:
#Verify geographic coverage and calculate height statistics
import numpy as np
from rasterio.warp import transform_bounds

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height_normalized = src.read(2, masked=True)

    height_m = height_normalized * 100

    geographic_bounds = transform_bounds(
        src.crs, "EPSG:4326", *src.bounds
    )

    print("Geographic bounds (WGS84):")
    print(geographic_bounds)

    print("\nBuilding density:")
    print("Mean:", round(float(density.mean()), 4))
    print("Maximum:", round(float(density.max()), 4))

    print("\nEstimated building height (metres):")
    print("Mean:", round(float(height_m.mean()), 2))
    print("Maximum:", round(float(height_m.max()), 2))
    print("Median:", round(float(np.ma.median(height_m)), 2))

    print("\nValid raster coverage:")
    print(f"{density.count():,} pixels")

In [ ]:
# Calculate VUI i.e. \[VUI = Building Density * Estimated Height
import numpy as np
import rasterio

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height_m = src.read(2, masked=True) * 100

    vui = density * height_m

    print("Vertical Urban Intensity (VUI)")
    print("Mean:", round(float(vui.mean()), 3))
    print("Maximum:", round(float(vui.max()), 3))
    print("Median:", round(float(np.ma.median(vui)), 3))

    built_cells = (density > 0) & (height_m > 0)

    print("\nCells with mapped buildings:", int(built_cells.sum()))
    print("Percentage of valid cells:",
          round(float(built_cells.sum() / density.count() * 100), 2), "%")

In [ ]:
#Export VUI GeoTIFF
import rasterio
import numpy as np

VUI_PATH = PROJECT_DIR / "Lusaka_Vertical_Urban_Intensity.tif"

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height_m = src.read(2, masked=True) * 100

    vui = density * height_m

    output = vui.filled(-9999).astype("float32")

    profile = src.profile.copy()
    profile.update(
        count=1,
        dtype="float32",
        nodata=-9999,
        compress="lzw"
    )

    with rasterio.open(VUI_PATH, "w", **profile) as dst:
        dst.write(output, 1)
        dst.set_band_description(1, "Vertical Urban Intensity")

print("VUI raster saved:", VUI_PATH)

In [ ]:
# Create the spatial grid
import rasterio
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

records = []

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height = src.read(2, masked=True) * 100

    for row in range(src.height):
        for col in range(src.width):
            d = density[row, col]
            h = height[row, col]

            if np.ma.is_masked(d) or np.ma.is_masked(h):
                continue

            if d == 0 or h == 0:
                continue

            x_left, y_bottom, x_right, y_top = rasterio.transform.array_bounds(
                1, 1, src.window_transform(
                    rasterio.windows.Window(col, row, 1, 1)
                )
            )

            records.append({
                "density": float(d),
                "height_m": float(h),
                "vui": float(d * h),
                "geometry": box(x_left, y_bottom, x_right, y_top)
            })

grid = gpd.GeoDataFrame(records, geometry="geometry", crs=src.crs)

print("Grid cells:", len(grid))
print("CRS:", grid.crs)
print(grid.head())

In [ ]:
# Install PostgreSQL and PostGIS
import subprocess
import re

print("Updating package lists...")
subprocess.run(["apt-get", "update", "-qq"], check=True)

print("Installing PostgreSQL and PostGIS...")
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "postgresql", "postgis"],
    check=True
)

version = subprocess.check_output(
    ["pg_config", "--version"], text=True
)
major_version = re.search(r"\d+", version).group()

print("PostgreSQL version:", version.strip())

subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     f"postgresql-{major_version}-postgis-3"],
    check=True
)

print("PostgreSQL and PostGIS installation completed.")

In [ ]:
# Start PostgreSQL and create the project database
import subprocess

subprocess.run(
    ["service", "postgresql", "start"],
    check=True
)

subprocess.run(
    ["su", "postgres", "-c",
     "createdb geotwin_lusaka"],
    check=False
)

subprocess.run(
    ["su", "postgres", "-c",
     "psql -d geotwin_lusaka -c 'CREATE EXTENSION IF NOT EXISTS postgis;'"],
    check=True
)

print("GeoTwin Lusaka database created.")

In [ ]:
# Connect Python to PostGIS and import the grid
import secrets
import subprocess
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from geoalchemy2 import Geometry

DB_USER = "geotwin"
DB_PASSWORD = secrets.token_hex(16)
DB_NAME = "geotwin_lusaka"

# Create a temporary database user
subprocess.run(
    ["su", "postgres", "-c",
     f"psql -d {DB_NAME} -c \"CREATE ROLE {DB_USER} LOGIN PASSWORD '{DB_PASSWORD}';\""],
    check=True
)

# Grant database permissions
subprocess.run(
    ["su", "postgres", "-c",
     f"psql -d {DB_NAME} -c \"GRANT ALL ON SCHEMA public TO {DB_USER};\""],
    check=True
)

# Connect to PostGIS
connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database=DB_NAME
)

engine = create_engine(connection_url)

# Import spatial grid
grid.to_postgis(
    name="building_intensity_grid",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=2000
)

print("Grid successfully imported into PostGIS.")

In [ ]:
# Run PostGIS spatial queries
from sqlalchemy import text

queries = {
    "Total grid cells": """
        SELECT COUNT(*) FROM building_intensity_grid;
    """,

    "Average building density": """
        SELECT ROUND(AVG(density)::numeric, 4)
        FROM building_intensity_grid;
    """,

    "Maximum estimated height (m)": """
        SELECT ROUND(MAX(height_m)::numeric, 2)
        FROM building_intensity_grid;
    """,

    "Maximum VUI": """
        SELECT ROUND(MAX(vui)::numeric, 3)
        FROM building_intensity_grid;
    """,

    "Geometry SRID": """
        SELECT ST_SRID(geometry)
        FROM building_intensity_grid
        LIMIT 1;
    """,

    "Spatial extent": """
        SELECT ST_AsText(ST_Extent(geometry))
        FROM building_intensity_grid;
    """
}

with engine.connect() as connection:
    for label, query in queries.items():
        result = connection.execute(text(query)).scalar()
        print(f"{label}: {result}")

In [ ]:
# Generate interactive 3D visualization
import plotly.express as px

# Sample cells for efficient 3D rendering
sample = grid.sample(
    n=min(15000, len(grid)),
    random_state=42
).copy()

# Calculate cell centroids
sample["x"] = sample.geometry.centroid.x
sample["y"] = sample.geometry.centroid.y

fig = px.scatter_3d(
    sample,
    x="x",
    y="y",
    z="height_m",
    color="vui",
    color_continuous_scale="Viridis",
    hover_data=["density", "height_m", "vui"],
    title="GeoTwin Lusaka: 3D Building Intensity"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Easting (m)",
        yaxis_title="Northing (m)",
        zaxis_title="Estimated Height (m)"
    ),
    height=750
)

HTML_PATH = PROJECT_DIR / "GeoTwin_Lusaka_3D_Visualization.html"
fig.write_html(HTML_PATH)

fig.show()

print("3D visualization saved:", HTML_PATH)

In [ ]:
#Export the three maps
import matplotlib.pyplot as plt
import rasterio
import numpy as np

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height = src.read(2, masked=True) * 100

with rasterio.open(VUI_PATH) as src:
    vui = src.read(1, masked=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

maps = [
    (density, "Building Density", "Density"),
    (height, "Estimated Building Height (m)", "Height (m)"),
    (vui, "Vertical Urban Intensity", "VUI")
]

for ax, (data, title, legend) in zip(axes, maps):
    image = ax.imshow(data, cmap="viridis")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(image, ax=ax, shrink=0.7, label=legend)

plt.suptitle("GeoTwin Lusaka | Building Intelligence", fontsize=16)
plt.tight_layout()

MAP_PATH = PROJECT_DIR / "GeoTwin_Lusaka_Three_Maps.png"
plt.savefig(MAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Maps saved:", MAP_PATH)


In [ ]:
# Improve the height and VUI map contrast
import matplotlib.pyplot as plt
import rasterio
import numpy as np

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1, masked=True)
    height = src.read(2, masked=True) * 100

with rasterio.open(VUI_PATH) as src:
    vui = src.read(1, masked=True)

# Calculate display limits using positive values only
height_vmax = np.percentile(height.compressed()[height.compressed() > 0], 98)
vui_vmax = np.percentile(vui.compressed()[vui.compressed() > 0], 98)

fig, axes = plt.subplots(1, 3, figsize=(19, 7))

maps = [
    (density, "Building Density", "Density", None),
    (height, "Estimated Building Height", "Height (m)", height_vmax),
    (vui, "Vertical Urban Intensity", "VUI", vui_vmax)
]

for ax, (data, title, label, vmax) in zip(axes, maps):
    image = ax.imshow(data, cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
    fig.colorbar(image, ax=ax, shrink=0.75, label=label)

fig.suptitle("GeoTwin Lusaka | Building Intelligence", fontsize=17)
plt.tight_layout()

MAP_PATH = PROJECT_DIR / "GeoTwin_Lusaka_Three_Maps_Improved.png"
plt.savefig(MAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Height display maximum:", round(float(height_vmax), 2), "m")
print("VUI display maximum:", round(float(vui_vmax), 2))
print("Improved maps saved:", MAP_PATH)

In [ ]:
# Generate the project statistical summary
import numpy as np
import rasterio
from sqlalchemy import text

# Full raster statistics
with rasterio.open(RASTER_PATH) as src:
    density_all = src.read(1, masked=True)
    height_all = src.read(2, masked=True) * 100

    valid = (~np.ma.getmaskarray(density_all))
    built = valid & (density_all.data > 0) & (height_all.data > 0)

    vui_all = density_all.data * height_all.data

    print("===== FULL RASTER =====")
    print("Valid cells:", int(valid.sum()))
    print("Building cells:", int(built.sum()))
    print("Building coverage (%):", round(built.sum() / valid.sum() * 100, 2))
    print("Mean density:", round(float(density_all.mean()), 4))
    print("Mean height across valid cells (m):",
          round(float(height_all.mean()), 2))
    print("Mean VUI across valid cells:",
          round(float(vui_all[valid].mean()), 3))

# PostGIS statistics for building-containing cells
query = text("""
WITH stats AS (
    SELECT
        COUNT(*) AS cell_count,
        AVG(density) AS mean_density,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY density) AS median_density,
        MAX(density) AS max_density,
        AVG(height_m) AS mean_height,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY height_m) AS median_height,
        MAX(height_m) AS max_height,
        AVG(vui) AS mean_vui,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY vui) AS median_vui,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY vui) AS p90_vui,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY vui) AS p95_vui,
        MAX(vui) AS max_vui
    FROM building_intensity_grid
)
SELECT * FROM stats;
""")

with engine.connect() as connection:
    result = connection.execute(query).mappings().one()

    print("\n===== BUILDING-CONTAINING CELLS =====")
    for key, value in result.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

In [ ]:
# Export statistical results
import pandas as pd

statistics = pd.DataFrame([
    ["Valid raster cells", 1309564, "cells"],
    ["Building-containing cells", 378847, "cells"],
    ["Building coverage", 28.93, "%"],
    ["Mean density (all cells)", 0.0507, "fraction"],
    ["Mean density (building cells)", 0.1753, "fraction"],
    ["Median density (building cells)", 0.1379, "fraction"],
    ["Maximum density", 0.9668, "fraction"],
    ["Mean height (building cells)", 2.5479, "metres"],
    ["Median height (building cells)", 2.4000, "metres"],
    ["Maximum height", 36.1816, "metres"],
    ["Mean VUI (all cells)", 0.147, "index"],
    ["Mean VUI (building cells)", 0.5077, "index"],
    ["Median VUI (building cells)", 0.3309, "index"],
    ["90th percentile VUI", 1.0025, "index"],
    ["95th percentile VUI", 1.2927, "index"],
    ["Maximum VUI", 20.6172, "index"]
], columns=["Indicator", "Value", "Unit"])

STATS_PATH = PROJECT_DIR / "GeoTwin_Lusaka_Statistical_Results.csv"
statistics.to_csv(STATS_PATH, index=False)

print(statistics.to_string(index=False))
print("\nSaved:", STATS_PATH)

In [ ]:
#Analysis 1 —morphology classification

# Use median density and 90th-percentile height
density_threshold = grid["density"].quantile(0.50)
height_threshold = grid["height_m"].quantile(0.90)

# Use a consistent threshold for all cells
grid["morphology"] = np.select(
    [
        (grid["density"] >= density_threshold) &
        (grid["height_m"] >= height_threshold),

        (grid["density"] >= density_threshold) &
        (grid["height_m"] < height_threshold),

        (grid["density"] < density_threshold) &
        (grid["height_m"] >= height_threshold),

        (grid["density"] < density_threshold) &
        (grid["height_m"] < height_threshold)
    ],
    [
        "Dense-Taller",
        "Dense-Lower",
        "Sparse-Taller",
        "Sparse-Lower"
    ],
    default="Unclassified"
)

summary = (
    grid.groupby("morphology")
    .size()
    .rename("cell_count")
    .reset_index()
)

summary["percentage"] = (
    summary["cell_count"] / len(grid) * 100
).round(2)

print("Density threshold:", density_threshold)
print("Height threshold:", height_threshold)

print("\nFINAL MORPHOLOGY RESULTS")
print(summary.to_string(index=False))

print("\nUnclassified cells:",
      (grid["morphology"] == "Unclassified").sum())

In [ ]:
# Examine the height–density relationship
# Cross-tabulation of morphology categories
cross_tab = pd.crosstab(
    grid["morphology"],
    columns="Cell count"
)

print("Morphology cross-tabulation:")
print(cross_tab)

# Correlation between density and estimated height
print(
    "\nPearson correlation:",
    round(grid["density"].corr(grid["height_m"]), 4)
)

print(
    "Spearman correlation:",
    round(grid["density"].corr(grid["height_m"], method="spearman"), 4)
)

# Share of upper-height cells in each density group
upper_height = grid["height_m"] >= height_threshold

print(
    "\nUpper-height cells:",
    int(upper_height.sum())
)

print(
    "Upper-height cells in dense group (%):",
    round(
        (grid.loc[upper_height, "density"] >= density_threshold).mean() * 100,
        2
    )
)

In [ ]:
# Export the final morphology map and statistics
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

with rasterio.open(RASTER_PATH) as src:
    d = src.read(1, masked=True)
    h = src.read(2, masked=True) * 100

    valid = (
        ~np.ma.getmaskarray(d) &
        ~np.ma.getmaskarray(h) &
        (d.data > 0) &
        (h.data > 0)
    )

    classes = np.zeros(d.shape, dtype="uint8")

    classes[valid & (d.data < density_threshold) &
            (h.data < height_threshold)] = 1

    classes[valid & (d.data >= density_threshold) &
            (h.data < height_threshold)] = 2

    classes[valid & (d.data < density_threshold) &
            (h.data >= height_threshold)] = 3

    classes[valid & (d.data >= density_threshold) &
            (h.data >= height_threshold)] = 4

    profile = src.profile.copy()
    profile.update(count=1, dtype="uint8", nodata=0, compress="lzw")

    MORPH_PATH = PROJECT_DIR / "Lusaka_Building_Morphology.tif"

    with rasterio.open(MORPH_PATH, "w", **profile) as dst:
        dst.write(classes, 1)

# Recalculate category statistics
labels = {
    1: "Sparse-Lower",
    2: "Dense-Lower",
    3: "Sparse-Taller",
    4: "Dense-Taller"
}

counts = [
    {"morphology": name, "cell_count": int((classes == code).sum())}
    for code, name in labels.items()
]

summary = pd.DataFrame(counts)
summary["percentage"] = (
    summary["cell_count"] / valid.sum() * 100
).round(3)

summary.to_csv(
    PROJECT_DIR / "Lusaka_Building_Morphology_Statistics.csv",
    index=False
)

# Plot
cmap = ListedColormap([
    "#440154", "#21918c", "#5ec962", "#fde725"
])

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(np.ma.masked_equal(classes, 0), cmap=cmap, vmin=1, vmax=4)
ax.set_title("Lusaka | Building Morphology Typology", fontsize=16)
ax.axis("off")

legend = [
    Patch(color=cmap(i), label=labels[i + 1])
    for i in range(4)
]
ax.legend(handles=legend, loc="lower left")

plt.tight_layout()

MORPH_MAP = PROJECT_DIR / "Lusaka_Building_Morphology_Map.png"
plt.savefig(MORPH_MAP, dpi=300, bbox_inches="tight")
plt.show()

print(summary.to_string(index=False))
print("\nFinal morphology outputs saved.")

In [ ]:
# Spatial hotspot detection

#Create the approximately 1 km analytical grid
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
from affine import Affine
from shapely.geometry import Polygon

BLOCK = 26  # 26 × 38.2 m ≈ 993.7 m

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1)
    height = src.read(2) * 100
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    valid = (
        (density != nodata) &
        (height != nodata) &
        np.isfinite(density) &
        np.isfinite(height)
    )

    vui = density * height

    rows, cols = vui.shape
    out_rows = int(np.ceil(rows / BLOCK))
    out_cols = int(np.ceil(cols / BLOCK))

    # Pad arrays to complete blocks
    pad_rows = out_rows * BLOCK - rows
    pad_cols = out_cols * BLOCK - cols

    vui_padded = np.pad(
        np.where(valid, vui, 0),
        ((0, pad_rows), (0, pad_cols))
    )

    valid_padded = np.pad(
        valid.astype(np.uint8),
        ((0, pad_rows), (0, pad_cols))
    )

    vui_blocks = vui_padded.reshape(
        out_rows, BLOCK, out_cols, BLOCK
    ).sum(axis=(1, 3))

    count_blocks = valid_padded.reshape(
        out_rows, BLOCK, out_cols, BLOCK
    ).sum(axis=(1, 3))

    mean_vui = np.divide(
        vui_blocks,
        count_blocks,
        out=np.full_like(vui_blocks, np.nan, dtype=float),
        where=count_blocks > 0
    )

    # Create approximately 1 km polygons
    records = []

    for r in range(out_rows):
        for c in range(out_cols):
            if count_blocks[r, c] == 0:
                continue

            cell_transform = (
                transform * Affine.translation(c * BLOCK, r * BLOCK)
            )

            corners = [
                cell_transform * (0, 0),
                cell_transform * (BLOCK, 0),
                cell_transform * (BLOCK, BLOCK),
                cell_transform * (0, BLOCK)
            ]

            records.append({
                "mean_vui": float(mean_vui[r, c]),
                "valid_pixels": int(count_blocks[r, c]),
                "geometry": Polygon(corners)
            })

grid_1km = gpd.GeoDataFrame(records, crs=crs)

print("Approximately 1 km grid cells:", len(grid_1km))
print("Mean neighbourhood VUI:", round(grid_1km["mean_vui"].mean(), 4))
print("Maximum neighbourhood VUI:", round(grid_1km["mean_vui"].max(), 4))
print("Grid CRS:", grid_1km.crs)

print("\nFirst five records:")
print(grid_1km.head())

In [ ]:
# Analysis 2 —Getis–Ord Gi* hotspot analysis

# Prepare spatial weights
!pip -q install libpysal esda statsmodels

import libpysal
from libpysal.weights import Queen

# Build neighbourhood relationships using shared polygon boundaries
grid_1km = grid_1km.reset_index(drop=True)

w = Queen.from_dataframe(
    grid_1km,
    use_index=False
)

# Binary spatial weights
w.transform = "B"

print("Grid cells:", len(grid_1km))
print("Spatial neighbours:", w.n)
print("Isolated cells:", len(w.islands))
print("Average neighbours per cell:", round(w.mean_neighbors, 2))

In [ ]:
# Calculate statistically significant hotspots
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests
from libpysal.weights import Queen
import numpy as np
import pandas as pd

# Exclude the single isolated cell
islands = list(w.islands)

grid_hot = grid_1km.drop(index=islands).copy()
grid_hot = grid_hot.reset_index(drop=True)

# Rebuild spatial weights for the retained cells
w_hot = Queen.from_dataframe(
    grid_hot,
    use_index=False
)
w_hot.transform = "B"

print("Cells included:", len(grid_hot))
print("Remaining isolated cells:", len(w_hot.islands))
print("Disconnected components:", w_hot.n_components)

# Calculate Getis-Ord Gi*
gi = G_Local(
    grid_hot["mean_vui"].values,
    w_hot,
    transform="B",
    permutations=999,
    star=True
)

# Multiple-testing correction
reject, p_adjusted, _, _ = multipletests(
    gi.p_sim,
    alpha=0.05,
    method="fdr_bh"
)

grid_hot["Gi_Z"] = gi.Zs
grid_hot["Gi_p"] = gi.p_sim
grid_hot["Gi_p_adjusted"] = p_adjusted
grid_hot["significant"] = reject

# Classify statistically significant hotspots and coldspots
grid_hot["hotspot_class"] = "Not significant"

grid_hot.loc[
    reject & (gi.Zs > 0),
    "hotspot_class"
] = "Hotspot"

grid_hot.loc[
    reject & (gi.Zs < 0),
    "hotspot_class"
] = "Coldspot"

print("\nHOTSPOT RESULTS")
print(grid_hot["hotspot_class"].value_counts())

print("\nZ-score range:")
print(round(float(np.min(gi.Zs)), 3),
      "to",
      round(float(np.max(gi.Zs)), 3))

print("\nAdjusted significant cells:", int(reject.sum()))

In [ ]:
# Create the hotspot map and export results
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Save hotspot statistics
HOTSPOT_CSV = PROJECT_DIR / "Lusaka_VUI_Hotspot_Statistics.csv"
HOTSPOT_GPKG = PROJECT_DIR / "Lusaka_VUI_Hotspot_Analysis.gpkg"
HOTSPOT_MAP = PROJECT_DIR / "Lusaka_VUI_Hotspot_Map.png"

grid_hot.drop(columns="geometry").to_csv(
    HOTSPOT_CSV, index=False
)

grid_hot.to_file(
    HOTSPOT_GPKG,
    layer="vui_hotspots",
    driver="GPKG"
)

# Plot hotspot map
fig, ax = plt.subplots(figsize=(12, 10))

colors = {
    "Hotspot": "#d7191c",
    "Coldspot": "#2c7bb6",
    "Not significant": "#d9d9d9"
}

for category, color in colors.items():
    subset = grid_hot[
        grid_hot["hotspot_class"] == category
    ]

    if not subset.empty:
        subset.plot(
            ax=ax,
            color=color,
            edgecolor="none",
            label=category
        )

ax.set_title(
    "Lusaka | Building Intensity Hotspot Analysis (Getis-Ord Gi*)",
    fontsize=14
)
ax.set_axis_off()

legend_items = [
    Patch(facecolor=color, label=category)
    for category, color in colors.items()
]

ax.legend(
    handles=legend_items,
    loc="lower left",
    frameon=True,
    title="Spatial significance"
)

plt.tight_layout()
plt.savefig(HOTSPOT_MAP, dpi=300, bbox_inches="tight")
plt.show()

print("Hotspot map:", HOTSPOT_MAP)
print("Statistics:", HOTSPOT_CSV)
print("Spatial data:", HOTSPOT_GPKG)

In [ ]:
#Analysis 3- Neighbourhood-scale building intelligence
import numpy as np
import pandas as pd
import rasterio

BLOCK = 26

density_means = []
height_means = []
vui_means = []

with rasterio.open(RASTER_PATH) as src:
    density = src.read(1)
    height = src.read(2) * 100

    valid = (
        (density != src.nodata) &
        (height != src.nodata) &
        np.isfinite(density) &
        np.isfinite(height)
    )

    rows, cols = density.shape

    for r in range(0, rows, BLOCK):
        for c in range(0, cols, BLOCK):

            d = density[r:r+BLOCK, c:c+BLOCK]
            h = height[r:r+BLOCK, c:c+BLOCK]
            v = valid[r:r+BLOCK, c:c+BLOCK]

            if v.sum() == 0:
                continue

            density_means.append(float(d[v].mean()))
            height_means.append(float(h[v].mean()))
            vui_means.append(float((d[v] * h[v]).mean()))

# Confirm alignment with the existing 1 km grid
assert len(grid_1km) == len(vui_means), (
    f"Grid mismatch: {len(grid_1km)} polygons vs "
    f"{len(vui_means)} raster blocks"
)

grid_1km["mean_density"] = density_means
grid_1km["mean_height_m"] = height_means
grid_1km["mean_vui"] = vui_means

# Rank grid units by mean VUI
grid_1km["vui_rank"] = (
    grid_1km["mean_vui"]
    .rank(method="min", ascending=False)
    .astype(int)
)

top20 = (
    grid_1km.drop(columns="geometry")
    .sort_values("mean_vui", ascending=False)
    .head(20)
)

print("NEIGHBOURHOOD-SCALE SUMMARY")
print(
    grid_1km[
        ["mean_density", "mean_height_m", "mean_vui"]
    ].describe().round(4)
)

print("\n20 HIGHEST-INTENSITY GRID UNITS")
print(
    top20[
        ["vui_rank", "mean_density", "mean_height_m", "mean_vui"]
    ].to_string(index=False)
)

print("\nGrid units analysed:", len(grid_1km))

In [ ]:
# Create the neighbourhood intensity map
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Five relative intensity groups
grid_1km["intensity_class"] = pd.qcut(
    grid_1km["mean_vui"].rank(method="first"),
    q=5,
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

# Identify top 5% of grid units
top_threshold = grid_1km["mean_vui"].quantile(0.95)
grid_1km["top_5_percent"] = (
    grid_1km["mean_vui"] >= top_threshold
)

print("Top 5% VUI threshold:", round(top_threshold, 4))
print("Top 5% grid units:", int(grid_1km["top_5_percent"].sum()))

# Summarise intensity groups
class_summary = (
    grid_1km.groupby("intensity_class", observed=False)
    .agg(
        cell_count=("mean_vui", "size"),
        average_vui=("mean_vui", "mean"),
        average_density=("mean_density", "mean"),
        average_height_m=("mean_height_m", "mean")
    )
    .round(4)
)

print("\nNEIGHBOURHOOD INTENSITY GROUPS")
print(class_summary)

# Export data
grid_1km.drop(columns="geometry").to_csv(
    PROJECT_DIR / "Lusaka_Neighbourhood_Intensity_Statistics.csv",
    index=False
)

grid_1km.to_file(
    PROJECT_DIR / "Lusaka_Neighbourhood_Intensity.gpkg",
    layer="neighbourhood_intensity",
    driver="GPKG"
)

# Plot map
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

colors = [
    "#440154",
    "#3b528b",
    "#21918c",
    "#5ec962",
    "#fde725"
]

fig, ax = plt.subplots(figsize=(12, 9))

grid_1km.plot(
    column="intensity_class",
    cmap=ListedColormap(colors),
    linewidth=0,
    legend=True,
    ax=ax,
    legend_kwds={
        "loc": "lower left",
        "title": "Mean VUI"
    }
)

ax.set_title(
    "Lusaka | Neighbourhood-Scale Building Intensity",
    fontsize=15
)

ax.set_axis_off()
plt.tight_layout()

NEIGHBOURHOOD_MAP = (
    PROJECT_DIR / "Lusaka_Neighbourhood_Building_Intensity.png"
)

plt.savefig(
    NEIGHBOURHOOD_MAP,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Map saved:", NEIGHBOURHOOD_MAP)

In [ ]:
# Integrating the analysis into PostGIS

# Import analytical layers into PostGIS
from sqlalchemy import text

# Import neighbourhood intensity layer
grid_1km.to_postgis(
    name="lusaka_neighbourhood_intensity",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

print("Neighbourhood intensity layer imported.")

# Import hotspot analysis layer
grid_hot.to_postgis(
    name="lusaka_vui_hotspots",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

print("Hotspot analysis layer imported.")

# Create spatial indexes
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS
        idx_lusaka_neighbourhood_geom
        ON lusaka_neighbourhood_intensity
        USING GIST (geometry)
    """))

    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS
        idx_lusaka_hotspots_geom
        ON lusaka_vui_hotspots
        USING GIST (geometry)
    """))

print("\nSpatial indexes created.")

# Verify imported records
with engine.connect() as conn:
    for table in [
        "building_intensity_grid",
        "lusaka_neighbourhood_intensity",
        "lusaka_vui_hotspots"
    ]:
        count = conn.execute(
            text(f'SELECT COUNT(*) FROM "{table}"')
        ).scalar()

        print(f"{table}: {count:,} records")

In [ ]:
# Run spatial SQL analysis
import pandas as pd
from sqlalchemy import text

# Query 1: Neighbourhood intensity summary
q1 = """
SELECT
    intensity_class,
    COUNT(*) AS grid_cells,
    ROUND(AVG(mean_vui)::numeric, 4) AS average_vui,
    ROUND(AVG(mean_density)::numeric, 4) AS average_density,
    ROUND(AVG(mean_height_m)::numeric, 4) AS average_height_m
FROM lusaka_neighbourhood_intensity
GROUP BY intensity_class
ORDER BY average_vui;
"""

# Query 2: Spatial join between intensity and hotspot layers
q2 = """
SELECT
    n.intensity_class,
    h.hotspot_class,
    COUNT(*) AS cell_count
FROM lusaka_neighbourhood_intensity n
JOIN lusaka_vui_hotspots h
    ON ST_Equals(n.geometry, h.geometry)
GROUP BY n.intensity_class, h.hotspot_class
ORDER BY n.intensity_class, h.hotspot_class;
"""

# Query 3: Ten highest-intensity grid units
q3 = """
SELECT
    vui_rank,
    ROUND(mean_density::numeric, 4) AS mean_density,
    ROUND(mean_height_m::numeric, 3) AS mean_height_m,
    ROUND(mean_vui::numeric, 4) AS mean_vui,
    intensity_class
FROM lusaka_neighbourhood_intensity
ORDER BY mean_vui DESC
LIMIT 10;
"""

with engine.connect() as conn:
    print("QUERY 1 — INTENSITY CLASS SUMMARY")
    display(pd.read_sql(text(q1), conn))

    print("\nQUERY 2 — SPATIAL HOTSPOT CROSS-TABULATION")
    display(pd.read_sql(text(q2), conn))

    print("\nQUERY 3 — TOP 10 HIGH-INTENSITY GRID UNITS")
    display(pd.read_sql(text(q3), conn))

In [ ]:
# Calculate percentages and validate the results
from sqlalchemy import text
import pandas as pd

query = """
SELECT
    n.intensity_class,
    COUNT(*) AS total_cells,
    COUNT(*) FILTER (
        WHERE h.hotspot_class = 'Hotspot'
    ) AS hotspots,
    COUNT(*) FILTER (
        WHERE h.hotspot_class = 'Coldspot'
    ) AS coldspots,
    COUNT(*) FILTER (
        WHERE h.hotspot_class = 'Not significant'
    ) AS not_significant,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE h.hotspot_class = 'Hotspot'
        ) / COUNT(*), 2
    ) AS hotspot_percentage,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE h.hotspot_class = 'Coldspot'
        ) / COUNT(*), 2
    ) AS coldspot_percentage
FROM lusaka_neighbourhood_intensity n
LEFT JOIN lusaka_vui_hotspots h
    ON ST_Equals(n.geometry, h.geometry)
GROUP BY n.intensity_class
ORDER BY AVG(n.mean_vui);
"""

with engine.connect() as conn:
    result = pd.read_sql(text(query), conn)

display(result)

print("\nTotal neighbourhood cells:", result["total_cells"].sum())
print("Total cells with hotspot results:",
      result[["hotspots", "coldspots", "not_significant"]].sum().sum())